<a href="https://colab.research.google.com/github/1mmagg1e/IB-Math-calculator-Assistant/blob/main/SSU_Data_Dash_Shared_Data_Preparation_Steps_1_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SSU Data Dash 2026 — Shared Data Preparation (Steps 1–9)
**团队共享版 · Google Colab · List-aware 基础实验准备**

**使用方法：** 上传本 `.ipynb` 到 [Google Colab](https://colab.research.google.com/)，选择 **运行时 → 全部运行**。每次重新打开 Colab 都可以从头运行，不依赖上一次会话。首次读取数据需要网络。

**目标：** 共同检查、整理和记录数据，为 Regression / Decision Tree 提供一致的基础特征和时间验证样本。**此 Notebook 不训练模型，也不宣称已完成正式比赛验证。**

**重要规则：** 不删除原始数据、不自动剔除极端高价房；训练目标 `sold_price` 不能进入特征；测试时未知的 `date_sold`、`days_on_market` 不作为特征；任何从数据学习的填补、编码、缩放必须在每轮训练集上拟合。

## Step 1 — 加载官方数据（Data Loading）
Colab 的临时磁盘会重置，所以每次新会话需要重新 clone 和 load。下方会自动检查文件夹是否已存在，避免重复 clone。

In [ ]:
import os, subprocess, sys
from pathlib import Path
import pandas as pd
import numpy as np

REPO = Path('/content/SSU-Data-Dash')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/erwinzhang7/SSU-Data-Dash.git', str(REPO)], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

try:
    from data import load
    listings = load('listings')
except ImportError:
    # 仅在缺少官方项目依赖时安装；避免无必要地覆盖 Colab 内置包
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    from data import load
    listings = load('listings')

print('Dataset shape:', listings.shape)
print('Columns:', listings.columns.tolist())
display(listings.head(3))

listings.parquet: reconstructing file:   0%|          |  0.00B / 32.9MB            

listings.parquet: downloading bytes:           |  0.00B            

Dataset shape: (150000, 22)
Columns: ['ml_num', 'sold_price', 'list_price', 'date_listed', 'date_sold', 'days_on_market', 'house_type_name', 'house_style', 'bedrooms', 'washrooms', 'parking_total', 'garage', 'house_area', 'lot_front', 'lot_depth', 'municipality', 'community', 'latitude', 'longitude', 'external_avm_estimate', 'external_avm_estimate_date', 'remarks']


,ml_num,sold_price,list_price,date_listed,date_sold,days_on_market,house_type_name,house_style,bedrooms,washrooms,...,house_area,lot_front,lot_depth,municipality,community,latitude,longitude,external_avm_estimate,external_avm_estimate_date,remarks
0,SYN-0023247,364100.0,371200.0,2020-12-29,2021-01-01,3,Condo Apt,Apartment,1,1,...,622,NaN,NaN,Toronto,Sumacpark,43.602039,-79.261184,366200.0,2021-02-14,Looking For More Space In A Turnkey Condo? Thi...
1,SYN-0156988,599000.0,595800.0,2020-12-26,2021-01-01,6,Condo Apt,Apartment,2,1,...,649,NaN,NaN,Toronto,Hickorymills,43.695388,-79.510865,622800.0,2021-12-26,"Imagine a quiet evening on Sequoia Trail, wher..."
2,SYN-0087706,492900.0,462200.0,2020-12-22,2021-01-01,10,Condo Apt,Apartment,2,1,...,900-1099,NaN,NaN,North York,Hazelmills,43.740512,-79.386216,NaN,2020-12-03,"Discover this inviting 2-bedroom, 1-bath condo..."


## Step 2 — 数据概况（Data Inspection）
查看类型、缺失数量和比例、统计量；这是**检查**，不代表已经处理缺失。

In [ ]:
print('=== Data types ===')
display(listings.dtypes.rename('dtype').to_frame())
missing = pd.DataFrame({
    'missing_count': listings.isna().sum(),
    'missing_pct': listings.isna().mean().mul(100).round(2)
}).sort_values('missing_pct', ascending=False)
display(missing)
display(listings[['sold_price','list_price']].describe())

=== Data types ===


,dtype
ml_num,object
sold_price,float64
list_price,float64
date_listed,datetime64[us]
date_sold,datetime64[us]
days_on_market,int64
house_type_name,object
house_style,object
bedrooms,int64
washrooms,int64


,missing_count,missing_pct
lot_depth,87263,58.18
lot_front,87263,58.18
external_avm_estimate,1306,0.87
list_price,0,0.00
ml_num,0,0.00
sold_price,0,0.00
days_on_market,0,0.00
date_sold,0,0.00
date_listed,0,0.00
house_type_name,0,0.00


,sold_price,list_price
count,1.500000e+05,1.500000e+05
mean,1.073917e+06,1.074505e+06
std,8.125300e+05,8.104716e+05
min,1.726000e+05,1.618000e+05
25%,6.237000e+05,6.317000e+05
50%,8.566500e+05,8.597000e+05
75%,1.239800e+06,1.233900e+06
max,3.000000e+07,2.807800e+07


## Step 3 — 检查特征格式（Feature Inspection）
检查 `house_area`、房屋类型、风格。房屋面积有**具体数值**和**区间**两种格式。

In [ ]:
for col in ['house_area', 'house_type_name', 'house_style']:
    print(f'\n=== {col} ===')
    display(listings[col].astype(str).value_counts().head(20).rename('count').to_frame())


=== house_area ===


,count
house_area,
700-899,6282
600-799,6101
800-999,4298
900-1099,3934
1000-1199,3714
1100-1299,3648
1200-1399,3426
500-699,3422
1300-1499,3250



=== house_type_name ===


,count
house_type_name,
Condo Apt,76374
Detached,43202
Semi-Detached,14397
Condo Townhouse,10160
Freehold Townhouse,5138
Co-Op Apt,729



=== house_style ===


,count
house_style,
Apartment,72455
2-Storey,36357
Bungalow,13675
3-Storey,8962
2 1/2 Storey,3117
Stacked Townhse,2613
Loft,2538
Bungalow-Raised,2477
1 1/2 Storey,2247


## Step 4 — 房屋面积转换（House Area Conversion）
- `622` → `622.0`；`900–1099` → `999.5`。
- 面积区间的中点是近似值，不是真实精确面积。
- 保留原始 `house_area`，新建 `house_area_mid`；不直接删除转换失败的行。

In [ ]:
import re

def parse_house_area(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace('–','-').replace('—','-').replace('−','-').replace(',','')
    try:
        return float(s)
    except ValueError:
        pass
    match = re.fullmatch(r'\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*', s)
    if match:
        lo, hi = map(float, match.groups())
        return (lo + hi) / 2 if 0 < lo <= hi else np.nan
    return np.nan

df = listings.copy()
df['house_area_mid'] = df['house_area'].map(parse_house_area)
print('Area conversion missing:', df['house_area_mid'].isna().sum())
print('Rows preserved:', len(df))
display(df[['house_area','house_area_mid','house_type_name','list_price']].head(10))
if df['house_area_mid'].isna().any():
    print('Unparsed values:')
    display(df.loc[df['house_area_mid'].isna(), 'house_area'].astype(str).value_counts().head(20))

Area conversion missing: 0
Rows preserved: 150000


,house_area,house_area_mid,house_type_name,list_price
0,622,622.0,Condo Apt,371200.0
1,649,649.0,Condo Apt,595800.0
2,900-1099,999.5,Condo Apt,462200.0
3,876,876.0,Condo Apt,524900.0
4,514,514.0,Condo Apt,558300.0
5,1500-1699,1599.5,Detached,874900.0
6,562,562.0,Condo Apt,464400.0
7,1000-1199,1099.5,Condo Apt,871100.0
8,723,723.0,Condo Apt,525400.0
9,1000-1199,1099.5,Freehold Townhouse,615100.0


## Step 5 — 重复检查（Duplicate Check）
同时检查**完全重复行**与 `ml_num` 重复。重复编号未必意味着错误；这里**只标记，不删除**。

In [ ]:
dup_summary = pd.DataFrame([{
    'total_rows': len(df),
    'exact_duplicate_rows': int(df.duplicated().sum()),
    'duplicate_ml_num_after_first': int(df['ml_num'].duplicated().sum()),
    'unique_ml_num': int(df['ml_num'].nunique(dropna=True))
}])
display(dup_summary)
repeated = df.loc[df['ml_num'].duplicated(keep=False)].sort_values('ml_num')
print('Rows with repeated ml_num:', len(repeated))
display(repeated[['ml_num','date_listed','date_sold','list_price','sold_price']].head(15))

,total_rows,exact_duplicate_rows,duplicate_ml_num_after_first,unique_ml_num
0,150000,0,0,150000


Rows with repeated ml_num: 0


,ml_num,date_listed,date_sold,list_price,sold_price


## Step 6 — 异常值与有效性（Outlier & Validity Check）
检查无效价格、面积、房间数、日期先后，以及极端值。**极端值是待研究对象，不自动删除**。

In [ ]:
df['date_listed'] = pd.to_datetime(df['date_listed'], errors='coerce')
df['date_sold'] = pd.to_datetime(df['date_sold'], errors='coerce')
checks = {
    'sold_price <= 0': int((df['sold_price'] <= 0).sum()),
    'list_price <= 0': int((df['list_price'] <= 0).sum()),
    'house_area_mid <= 0': int((df['house_area_mid'] <= 0).sum()),
    'bedrooms < 0': int((df['bedrooms'] < 0).sum()),
    'washrooms < 0': int((df['washrooms'] < 0).sum()),
    'parking_total < 0': int((df['parking_total'] < 0).sum()),
    'date_sold < date_listed': int((df['date_sold'] < df['date_listed']).sum()),
    'invalid date_listed': int(df['date_listed'].isna().sum()),
    'invalid date_sold': int(df['date_sold'].isna().sum())
}
validity = pd.DataFrame({'check': list(checks), 'count': list(checks.values())})
display(validity)
for col in ['sold_price','list_price','house_area_mid','bedrooms','washrooms']:
    print(f'\n{col}:')
    display(df[col].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).to_frame().T)
print('Top 5 sold prices (inspection only):')
display(df.nlargest(5,'sold_price')[['ml_num','sold_price','list_price','house_area_mid','house_type_name']])

,check,count
0,sold_price <= 0,0
1,list_price <= 0,0
2,house_area_mid <= 0,0
3,bedrooms < 0,0
4,washrooms < 0,0
5,parking_total < 0,0
6,date_sold < date_listed,0
7,invalid date_listed,0
8,invalid date_sold,0



sold_price:


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
sold_price,150000.0,1073917.036,812529.953536,172600.0,356700.0,442700.0,623700.0,856650.0,1239800.0,2361410.0,4344403.0,30000000.0



list_price:


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
list_price,150000.0,1.074505e+06,810471.62734,161800.0,357300.0,446000.0,631700.0,859700.0,1233900.0,2354215.0,4366116.0,28078000.0


...

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
bedrooms,150000.0,2.341673,1.106252,0.0,0.0,1.0,1.0,2.0,3.0,4.0,5.0,6.0



washrooms:


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
washrooms,150000.0,2.158313,1.109122,1.0,1.0,1.0,1.0,2.0,3.0,4.0,6.0,6.0


Top 5 sold prices (inspection only):


,ml_num,sold_price,list_price,house_area_mid,house_type_name
58706,SYN-0003038,30000000.0,28078000.0,7085.0,Detached
50649,SYN-0074468,23145600.0,21276900.0,7999.5,Detached
80964,SYN-0134687,19842400.0,20037800.0,5629.0,Detached
149910,SYN-0090330,17035700.0,18203500.0,9000.0,Detached
54452,SYN-0054091,16984700.0,15372100.0,9000.0,Detached


## Step 7 — 缺失值策略（Missing Value Strategy）
**初版公共方案：** `lot_front`、`lot_depth` 缺失较多，基础模型暂不使用；`external_avm_estimate` 是否可用须先核对官方测试特征与估值时间，初版不用；`house_area_mid` 若仍缺失，在每轮模型 Pipeline 内用**训练集**中位数填补。分类变量也在 Pipeline 中处理。**这里不对全量数据计算中位数或 One-Hot 编码。**

In [ ]:
SHARED_NUMERIC_FEATURES = ['list_price','bedrooms','washrooms','parking_total','house_area_mid']
SHARED_CATEGORICAL_FEATURES = ['house_type_name','house_style']
SHARED_FEATURES = SHARED_NUMERIC_FEATURES + SHARED_CATEGORICAL_FEATURES
strategy = pd.DataFrame([
    ('lot_front / lot_depth','58% 左右缺失','基础版暂不使用；之后可按房型研究结构性缺失'),
    ('external_avm_estimate','少量缺失','先核实测试集可用性及时间戳；基础版不用'),
    ('house_area_mid','转换后检查','在训练 Pipeline 中填补；不可全数据先拟合'),
    ('house_type_name / house_style','类别型','在训练 Pipeline 中 One-Hot；未知类别忽略'),
    ('list_price','List-aware 特征','Off-market 绝对不能使用')
], columns=['feature','issue','shared initial policy'])
display(strategy)
print('Shared baseline features:', SHARED_FEATURES)

,feature,issue,shared initial policy
0,lot_front / lot_depth,58% 左右缺失,基础版暂不使用；之后可按房型研究结构性缺失
1,external_avm_estimate,少量缺失,先核实测试集可用性及时间戳；基础版不用
2,house_area_mid,转换后检查,在训练 Pipeline 中填补；不可全数据先拟合
3,house_type_name / house_style,类别型,在训练 Pipeline 中 One-Hot；未知类别忽略
4,list_price,List-aware 特征,Off-market 绝对不能使用


Shared baseline features: ['list_price', 'bedrooms', 'washrooms', 'parking_total', 'house_area_mid', 'house_type_name', 'house_style']


## Step 8 — 特征可用性 / Data Leakage 审核
**重要：** 以下是初步保守规则，最终必须以官方 handout / starter 中对两种测试集的字段说明为准。`sold_price` 是标签，不能做特征；`date_sold` 只能帮助确认训练标签何时已知；`days_on_market` 在挂牌时通常未知，先排除；`ml_num` 是标识符，先排除；`remarks` 暂不处理。

In [ ]:
availability = pd.DataFrame([
    ('sold_price','Target only','真实成交价，不能输入模型'),
    ('list_price','List-aware only','Off-market 禁用'),
    ('date_listed','Potential feature / splitting','可派生挂牌月份，但必须与官方预测时点一致'),
    ('date_sold','Split/filter only','预测时未知，防止泄漏'),
    ('days_on_market','Exclude initially','成交后才可能完整确定'),
    ('ml_num','Exclude','ID，可能造成记忆化'),
    ('external_avm_estimate','Hold / verify','核对测试集与估值日期后再决定'),
    ('remarks','Hold / verify','文本处理留到后续'),
    ('house_area_mid','Feature','从挂牌可知的面积格式转换'),
    ('house_type_name / house_style','Feature','分类特征')
], columns=['field','initial decision','reason'])
display(availability)
assert 'sold_price' not in SHARED_FEATURES
assert 'date_sold' not in SHARED_FEATURES
assert 'days_on_market' not in SHARED_FEATURES
print('Initial leakage checklist passed for baseline features.')

,field,initial decision,reason
0,sold_price,Target only,真实成交价，不能输入模型
1,list_price,List-aware only,Off-market 禁用
2,date_listed,Potential feature / splitting,可派生挂牌月份，但必须与官方预测时点一致
3,date_sold,Split/filter only,预测时未知，防止泄漏
4,days_on_market,Exclude initially,成交后才可能完整确定
5,ml_num,Exclude,ID，可能造成记忆化
6,external_avm_estimate,Hold / verify,核对测试集与估值日期后再决定
7,remarks,Hold / verify,文本处理留到后续
8,house_area_mid,Feature,从挂牌可知的面积格式转换
9,house_type_name / house_style,Feature,分类特征


Initial leakage checklist passed for baseline features.


## Step 9 — 统一时间验证协议（Shared Validation Protocol）
**共同规则：** 2025-12 至 2026-05 六个验证月；同一批验证房屋、同一预测目标、同一 MdAPE/MAPE；每月训练集仅包含该月第一天之前**已成交**的房屋。

下面按 `date_listed` 选择验证月房源（与之前的练习一致），同时按 `date_sold` 限制训练标签。**这只是团队的初版样本清单**：正式比赛必须对照官方 `run.py validate`，确认官方到底按挂牌月还是成交月定义验证样本、预测时点及可用字段；若不一致，应统一改为官方规则。**不能把它直接宣称为官方验证分数。**

⚠️ 六个月验证结果不能被无限次反复调参当作无偏测试；最后应预留独立的历史时间段或明确报告调参偏差。

In [ ]:
MONTH_STARTS = pd.date_range('2025-12-01','2026-05-01',freq='MS')
# 仅剔除明确无法用于监督训练/评估的记录；保留其他待检查异常
usable = df.loc[
    df['date_listed'].notna() & df['date_sold'].notna() &
    df['sold_price'].gt(0) & df['list_price'].gt(0)
].copy()

folds = {}
rows = []
for start in MONTH_STARTS:
    end = start + pd.DateOffset(months=1)
    train_idx = usable.index[usable['date_sold'] < start]
    valid_idx = usable.index[(usable['date_listed'] >= start) & (usable['date_listed'] < end)]
    folds[start.strftime('%Y-%m')] = {'train_idx': train_idx, 'valid_idx': valid_idx}
    rows.append({'validation_month':start.strftime('%Y-%m'),
                 'training_cutoff_exclusive':str(start.date()),
                 'train_rows':len(train_idx),'validation_rows':len(valid_idx),
                 'overlap_ids':len(set(train_idx).intersection(valid_idx))})
fold_summary = pd.DataFrame(rows)
display(fold_summary)
assert (fold_summary['overlap_ids'] == 0).all(), 'Training and validation overlap!'
print('Fold indices prepared. Review official validation protocol before modeling.')

,validation_month,training_cutoff_exclusive,train_rows,validation_rows,overlap_ids
0,2025-12,2025-12-01,139931,1085,0
1,2026-01,2026-01-01,141169,1293,0
2,2026-02,2026-02-01,142227,1447,0
3,2026-03,2026-03-01,143669,1856,0
4,2026-04,2026-04-01,145485,1798,0
5,2026-05,2026-05-01,147709,946,0


Fold indices prepared. Review official validation protocol before modeling.


## 统一评价函数：MdAPE / MAPE（供两组复用）
这是**工具函数**，此处没有训练任何模型。`MAPE` 是平均绝对百分比误差；`MdAPE` 是中位数绝对百分比误差。真实价格必须大于 0。

In [ ]:
def price_metrics(actual, predicted):
    a = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    if len(a) != len(p) or np.any(a <= 0) or not np.isfinite(a).all() or not np.isfinite(p).all():
        raise ValueError('Need same-length finite predictions and positive actual prices')
    ape = np.abs((a-p)/a)*100
    return {'MdAPE_pct':float(np.median(ape)), 'MAPE_pct':float(np.mean(ape))}

print('Example only:', price_metrics([100, 200], [90, 220]))

Example only: {'MdAPE_pct': 10.0, 'MAPE_pct': 10.0}


## 输出团队共享文件（Colab 本地 `/content/`）
导出：
1. `ssu_shared_prepared.csv`：保留所有原始行和新面积变量，**不是训练完毕的最终编码数据**。
2. `ssu_data_quality_checks.csv`：检查结果。
3. `ssu_walkforward_fold_summary.csv`：时间样本数。

Colab 临时磁盘会清空；需要保存时从左侧文件栏下载，或在最后一个单元格主动下载。

In [ ]:
OUT = Path('/content/ssu_shared_outputs')
OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT/'ssu_shared_prepared.csv', index=False)
validity.to_csv(OUT/'ssu_data_quality_checks.csv', index=False)
fold_summary.to_csv(OUT/'ssu_walkforward_fold_summary.csv', index=False)
missing.to_csv(OUT/'ssu_missingness.csv')
dup_summary.to_csv(OUT/'ssu_duplicate_summary.csv', index=False)
print('Saved files:')
for p in sorted(OUT.glob('*.csv')):
    print(' -',p.name, f'({p.stat().st_size/1024:.1f} KB)')

Saved files:
 - ssu_data_quality_checks.csv (0.2 KB)
 - ssu_duplicate_summary.csv (0.1 KB)
 - ssu_missingness.csv (0.4 KB)
 - ssu_shared_prepared.csv (83394.2 KB)
 - ssu_walkforward_fold_summary.csv (0.3 KB)


##  A / B / C / D：Next？
- **共同确认** Step 5 重复编号含义、Step 6 无效值、Step 7 缺失方案、Step 8 官方可用字段、Step 9 官方时间验证定义；有争议的规则先记录，**不要单方面删除**。
- **A / B（Regression）**：使用相同 `folds`、`SHARED_FEATURES` 和 `price_metrics()`，在每轮训练数据上单独拟合编码/填补；A 做基线，B 做变换、交互和误差分析。
- **C（Decision Tree）**：建立同一基础树，主要比较 `max_depth`；每轮记录训练与验证误差。
- **D（Decision Tree）**：建立同一基础树，并行研究 `min_samples_leaf`、`min_samples_split`、`ccp_alpha` 剪枝；每轮记录训练与验证误差。
- **C + D**：合并选出的**超参数**到同一棵树，再联合验证，检查过拟合；参数可能相互作用，需要联合微调。
- **两组**：对比同一验证样本上的 MdAPE/MAPE，之后才考虑融合 Regression / Tree 的预测。

**复现提醒：** 不要只共享 Colab 的运行截图；共享 Notebook、官方项目版本、规则说明和导出结果。Colab 重开后运行 **运行时 → 全部运行**。